In [1]:
# read json and save with indent 4
import json

with open("full_depth/camera_matrix.json", "r") as f:
    data = json.load(f)
    
with open("full_depth/camera_matrix.json", "w") as f:
    json.dump(data, f, indent=4)


### Sanity check
read 
- depth videos
- rgb videos
- timestamps

and compare them 

In [1]:
import os
import cv2
from pprint import pprint


ALL_CAMERAS = ["Aoutput0", "Aoutput1", "Aoutput2", "Aoutput3",
               "Boutput0", "Boutput1", "Boutput2", "Boutput3",
               "output0", "hololens"]

data_path = "/home/enerd/epfl-sk30-samples/2023_08_11_14_15_21"

frame_numbers = {"rgb": {}, "depth": {}}

for camera in ALL_CAMERAS:
    # read rgb video
    rgb_video_path = os.path.join(data_path, "videos", f"{camera}.mp4")
    rgb_video = cv2.VideoCapture(rgb_video_path)
    _, first_rgb_frame = rgb_video.read()
    frame_numbers["rgb"][camera] = int(rgb_video.get(cv2.CAP_PROP_FRAME_COUNT))

    if camera == "hololens":
        continue

    # read depth video
    depth_video_path = os.path.join(data_path, "videos_depth", f"{camera}.mp4")
    depth_video = cv2.VideoCapture(depth_video_path)
    _, first_depth_frame = depth_video.read()
    frame_numbers["depth"][camera] = int(depth_video.get(cv2.CAP_PROP_FRAME_COUNT))

    print(first_rgb_frame.shape, first_depth_frame.shape)

    #  depth timestamps
    with open(os.path.join(data_path, "timestamps_depth", f"{camera}.txt"), "r") as f:
        timestamps = f.readlines()
    frame_numbers["depth"][camera + "_timestamp"] = len(timestamps)


with open(os.path.join(data_path, "meta_data", "timestamps.txt"), "r") as f:
    timestamps = f.readlines()
frame_numbers["rgb"]["timestamp"] = len(timestamps)


pprint(frame_numbers)

(720, 1280, 3) (576, 640, 3)
(720, 1280, 3) (576, 640, 3)
(720, 1280, 3) (576, 640, 3)
(720, 1280, 3) (576, 640, 3)
(720, 1280, 3) (576, 640, 3)
(720, 1280, 3) (576, 640, 3)
(720, 1280, 3) (576, 640, 3)
(720, 1280, 3) (576, 640, 3)
(720, 1280, 3) (576, 640, 3)
{'depth': {'Aoutput0': 21463,
           'Aoutput0_timestamp': 21463,
           'Aoutput1': 14353,
           'Aoutput1_timestamp': 14353,
           'Aoutput2': 14346,
           'Aoutput2_timestamp': 14346,
           'Aoutput3': 14374,
           'Aoutput3_timestamp': 14374,
           'Boutput0': 21509,
           'Boutput0_timestamp': 21509,
           'Boutput1': 14421,
           'Boutput1_timestamp': 14421,
           'Boutput2': 14404,
           'Boutput2_timestamp': 14404,
           'Boutput3': 14383,
           'Boutput3_timestamp': 14383,
           'output0': 42956,
           'output0_timestamp': 42956},
 'rgb': {'Aoutput0': 38670,
         'Aoutput1': 38670,
         'Aoutput2': 38670,
         'Aoutput3': 38670

## Check camera distortion and registration

In [5]:
import json
import numpy as np
with open(os.path.join(data_path, "meta_data", "camera_matrix.json"), "r") as f:
    CAMERA_PARAMS = json.load(f)

pprint({k: np.array(v).shape for k, v in CAMERA_PARAMS["Aoutput0"].items()})

{'K': (3, 3),
 'depth_K': (3, 3),
 'depth_dist': (8,),
 'dist': (8,),
 'word2cam': (4, 4),
 'world2depth': (4, 4)}


In [8]:
mtx

array([[608.23376465,   0.        , 641.59838867],
       [  0.        , 608.10906982, 364.78900146],
       [  0.        ,   0.        ,   1.        ]])

In [7]:
import matplotlib.pyplot as plt
import numpy as np
import pyimof

dist = np.array(CAMERA_PARAMS["Aoutput0"]["dist"])
mtx = np.array(CAMERA_PARAMS["Aoutput0"]["K"])

rgb_video_path = os.path.join(data_path, "videos", "Aoutput0.mp4")
rgb_video = cv2.VideoCapture(rgb_video_path)

rgb_video.set(cv2.CAP_PROP_POS_FRAMES, 100)
_, rgb_frame = rgb_video.read()

h, w = rgb_frame.shape[:2]

newcameramtx, roi = cv2.getOptimalNewCameraMatrix(mtx, dist, (w,h), 0, (w,h))

dst = cv2.undistort(rgb_frame, mtx, dist, None, newcameramtx)

x, y, w, h = roi
# dst = dst[y:y+h, x:x+w]

print("undistorted.shape", dst.shape)
cv2.imwrite("samples/rgb.png", rgb_frame)
cv2.imwrite("samples/undistorted_rgb.png", dst)


# plt.subplot(131)
# plt.imshow(rgb_frame[:, :, ::-1])
# plt.subplot(132)
# plt.imshow(dst[:, :, ::-1])
# plt.subplot(133)
# plt.imshow(np.abs(rgb_frame[:, :, ::-1] - dst[:, :, ::-1]))
# plt.show()



# # calculate optical flow
# rgb_gray = cv2.cvtColor(rgb_frame, cv2.COLOR_BGR2GRAY)
# undistorted_gray = cv2.cvtColor(dst, cv2.COLOR_BGR2GRAY)

# assert rgb_gray.shape == undistorted_gray.shape, "RGB and undistorted frames must match"

# flow = cv2.calcOpticalFlowFarneback(
#     rgb_gray,
#     undistorted_gray,
#     None,
#     pyr_scale=0.5,
#     levels=3,
#     winsize=15,
#     iterations=3,
#     poly_n=5,
#     poly_sigma=1.2,
#     flags=0,
# )

# mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
hsv = np.zeros((*rgb_frame.shape[:2], 3), dtype=np.uint8)
# hsv[..., 0] = ang * 180 / np.pi / 2
hsv[..., 1] = 255
# hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
# optical_flow_vis = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

rgb_frame_gray = cv2.cvtColor(rgb_frame, cv2.COLOR_BGR2GRAY)
dst_gray = cv2.cvtColor(dst, cv2.COLOR_BGR2GRAY)

u, v = pyimof.solvers.tvl1(rgb_frame_gray, dst_gray)
mag, ang = cv2.cartToPolar(u, v)
hsv[...,0] = ang*180/np.pi/2
hsv[...,2] = cv2.normalize(mag,None,0,255,cv2.NORM_MINMAX)
optical_flow_vis = cv2.cvtColor(hsv,cv2.COLOR_HSV2BGR)


cv2.imwrite("samples/optical_flow.png", optical_flow_vis)



undistorted.shape (720, 1280, 3)


True

In [10]:
roi

(0, 0, 1279, 719)

In [11]:
dist = np.array(CAMERA_PARAMS["Aoutput0"]["depth_dist"])
mtx = np.array(CAMERA_PARAMS["Aoutput0"]["depth_K"])

depth_video_path = os.path.join(data_path, "videos_depth", "Aoutput0.mp4")
depth_video = cv2.VideoCapture(depth_video_path, cv2.CAP_FFMPEG)

depth_video.set(cv2.CAP_PROP_POS_FRAMES, 1069)
_, depth_frame = depth_video.read()

h, w = depth_frame.shape[:2]

newcameramtx, roi = cv2.getOptimalNewCameraMatrix(mtx, dist, (w,h), 0, (w,h))

dst = cv2.undistort(depth_frame, mtx, dist, None, newcameramtx)

# x, y, w, h = roi
# dst = dst[y:y+h, x:x+w]

print("undistorted.shape", dst.shape)

cv2.imwrite("samples/depth.png", depth_frame)
cv2.imwrite("samples/undistorted_depth.png", dst)



undistorted.shape (576, 640, 3)


True

In [ ]:
depth_frame.dtype

dtype('uint8')

In [11]:
640/576

1.1111111111111112

In [12]:
1280/720

1.7777777777777777

In [8]:
depth_frame.shape

(576, 640, 3)